Начинаем с монтирования google-диска:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import tensorflow as tf
import os
import random
import numpy as np
tf.keras.backend.set_floatx('float64')

seed_value= 10
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
tf.keras.utils.set_random_seed(seed_value)


Далее загрузим на диск датасет с сигнатурами подписи и импортируем данные в датафрейм.

In [3]:
import pandas as pd
df = pd.read_csv('путь к sign_dump.csv')

Далее опишем данные, используемые в данной части работы. В задании использован датасет, представляющий собой набор данных подписей пользователей, по несколько десятков на каждого пользователя, представленные в виде 144 параметров подписи, дающих полную информацию о динамике написания подписи. Датасет состоит из 2445 параметризированных подписей от 22 пользователей. В рамках датасета пользователь идентифицируется по уникальному для каждого пользователя номеру от 1 до 22 включительно. 
Подписи представлены значениями координат X, Y, Z, а также 2-мя углами наклона пера относительно плоскости планшета и значение давления пера на поверхность. Далее данные подписей был предобработаны методом Фурье и преобразованы в датасет из 144 параметров на подпись, по 8 гармоник на каждый параметр. Отбор классифицирующих признаков не производился.
Для симуляции работы нейронной сети пользователи разделены на легитимных, на основе которых будет происходить обучение сети, и незарегистрированных в системе, подписи которых система должна отвергать во время тестирования. 
Наборы подписей между собой не являются однородными, что в ходе исследования будет оказывать влияние на количество используемых наборов пользователей при обучении нейронных сетей. Количество подписей в наборах для каждого из пользователей неравномерно. 
Далее разделим датасет для будущего обучения:


In [ ]:
friend_users = [1,8,9,10,11,14,15,17,18,19,20,22]
alien_users = [2,3,4,5,6,7,12,16,21]

friend_df = df[df['id_user'].isin(friend_users)]
alien_df  = df[df['id_user'].isin(alien_users)]

alien_df['id_user']=0 

feed_df=friend_df.merge(alien_df,how='outer')

X = feed_df.drop(columns='id_user')
Y = feed_df['id_user']

from tensorflow.keras.utils import to_categorical
dummy_y = to_categorical(Y)
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,dummy_y)

#Поиск оптимальных гиперпараметров сети
Список параметров:



*  "Качество" и количество данных - в данном случае, под качеством подразумевается приближенность данных к "боевым" данным, т.к. любые выбросы и возмущения (исключения из правил) очень сильно влияют на сеть. Количество же важно потому, что чем больше данных - тем больше перед сетью будет примеров -> тем лучше будет подобрана f для y=f(x).
*   Исходное кодирование данных - от данного этапа будет зависеть, насколько эффективно будет происходить обучение. Часть кодирований не подходит для одного типа задач, но идеально подходит в другом. В первой работе мы выбрали Mean Square Error для определения пользователя, но, данный метод работал плохо, в связи с тем что для сети 6 пользователь выглядел как что-то между 5 и 7. В данном случае больше подходит категориальная кроссэнтропия - т.е. закодированная принадлежность значений определенному классу. 
*   Количество слоёв и нейронов в них, а также функции активации на каждом слое (т.н. архитектура сети) - данный пункт определяет, какова будет математическая модель нейронной сети.
*  Функция потерь. Чем меньше значение функции для сети - тем точнее она определяет результат. 
* Выбираемый оптимизатор, который будет определять "шаг" сети 
* Количество итераций обучения (эпох)

Каждый из этих параметров, при правильном выборе, повышает точность готовой сети. При этом важно не "переобучить" сеть - т.е. научить отличать тестовый в 100% случаях, но делать большой процент ошибок на значениях в валидационном датасете.

В случае визуализации переобучение выглядит так: график точности на тестовом датасете растёт, а на валидационном - падает.

In [5]:
# Импорт необходимых модулей, определение размера графиков

import pylab
pylab.rcParams['figure.figsize'] = (15, 10)
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GaussianNoise

Попробуем изменить функцию потерь на подходящую ситуации - категориальную кроссэнтропию.
Кроссэнтропия предполагает собой измерение, насколько уверена сеть, что данным признакам соответствует определенный вариант и не соответствуют все остальные.
Собственно, использовать для сигнатур подписи мы будем простейший персептрон. Соберем архитектуру нейронной сети:

In [6]:
model = Sequential()
model.add(Dense(10,activation='tanh',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

Далее для соберем сеть, указав функцию потерь, оптимизатор и основную метрику для оценивания - точность. Укажем, на каких данных сеть будет обучена и укажем, каким образом сохранить полученные веса.

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

Определим функции для вывода и отрисовки результатов:

In [ ]:
def display_accuracy():
    plt.plot(results.history['accuracy'])
    plt.plot(results.history['val_accuracy'])

    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Test'], loc='upper left')
    plt.show()

def display_loss():
    plt.plot(results.history['loss'])
    plt.plot(results.history['val_loss'])

    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Test'], loc='upper left')
    plt.show()

Выведем точность, затем - функцию потерь.

In [ ]:
display_accuracy()
display_loss()

Обратим внимание, что предыдущая сеть практически не обучалась. Такая ситуация достаточно обыденна, и это значит, что данная комбинация гиперпараметров абсолютно не подходит. Тем не менее, не будем возвращать обратно, а изменим другой параметр - функцию активации для первого слоя.

In [ ]:
model = Sequential()
model.add(Dense(10,activation='relu',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

Снова выведем результаты:

In [ ]:
display_accuracy()
display_loss()

Отлично. Порог в 50% точности пересечён.

Теперь попробуем добавить нейронов в слой

In [ ]:
model = Sequential()
# model.add(GaussianNoise(0.15,input_shape=(144,),name="noise_layer"))
model.add(Dense(30,activation='relu',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

Выведем точность и функцию потерь:

In [ ]:
display_accuracy()
display_loss()

80% порог пересечён. Продолжим.

Добавим слой зашумления данных. В данном случае он будет особенно эффективен, т.к. данных мало, но при этом в готовом виде сеть будет определять пользователя.

In [ ]:
model = Sequential()
model.add(GaussianNoise(0.15,input_shape=(144,),name="noise_layer"))
model.add(Dense(30,activation='relu',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

Снова выведем результаты:

In [ ]:
display_accuracy()
display_loss()

Попробуем переобучить сеть. Добавим больше слоёв и нейронов в них, уберем зашумление, и увеличим количество эпох обучения.

In [ ]:
model = Sequential()
model.add(Dense(300,activation='relu',input_shape=(144,)))
model.add(Dense(300,activation='relu',))
model.add(Dense(300,activation='relu',))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=1500)

model.save_weights('sign_model.h5')

И выведем результат этого обучения.

In [ ]:
display_accuracy()
display_loss()

Обратим внимание, что в процессе обучения сеть достигала точности до 95% на тестовом датасете, но к концу обучения точность упала до 90%, при том что на обучающем она стремилась к 100%.

Откатим изменения, добавленные для имитации переобучения.

In [ ]:
model = Sequential()
model.add(GaussianNoise(0.15,input_shape=(144,),name="noise_layer"))
model.add(Dense(30,activation='relu',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

Поскольку мы работаем с системой аутентификации необходимо добавить также модули для определения ошибок первого и второго рода, возникающих при попытке аутентификации с помощью подписи и порог, при котором система будет аутентифицировать пользователя.

Для этого сначала подключим библиотеку работы с математикой Numpy и импортируем функцию минимизации из библиотеки SciPy.

In [ ]:
import numpy as np
from scipy.optimize import minimize

После этого добавим следующие функции:

1.   Функцию Interpany, которая будет интерпретировать возникающие при тестировании сети ошибки как ошибки первого или второго рода и возвращать кортеж из двух переменных с процентом возникновения данных ошибок соответственно.
2.   Функцию GetMistSum, которая возвращает сумму ошибок первого и второго при указанном приоритете ошибки второго рода.
3.   Функцию FindTh, которая находит и возвращает порог принятия решения, при котором функция GetMistSum минимальна.

Для того, чтобы получить значение порога принятия решения и получившиеся ошибки первого и второго рода, добавим следующий код.

In [ ]:
def interpany(classouts,classets,classporog):
    classets = classets.applymap(bool)
    level = np.ones((classouts.shape[0],1))*classporog
    t2=classouts>level
    s1 = np.sum((t2==0)&classets)
    s2 = np.sum(t2&(~classets))
    ss1= np.sum(classets)
    p1=np.sum(s1)/np.sum(ss1)
    p2=np.sum(s2)/(t2.shape[0]*classets.shape[1]-np.sum(ss1))
    return np.array([p1,p2])
    
def GetMistSum(threahold, predc, truth, priority):
    first, second = interpany(predc, truth, threahold/100)
    return first + priority * second

def FindTh(predc, truth, priority):
    res = minimize(GetMistSum, 90, args=(predc, truth, priority),
                   method='powell',
                   options={'disp': True})
    return res.x / 100
target_df = feed_df.copy()
 
X = target_df.drop(columns='id_user')
Y = to_categorical(target_df['id_user'])
predc=pd.DataFrame(model.predict(X))
truth=pd.DataFrame(Y).applymap(round)
 
OptimalTh = FindTh(predc, truth, 10)
print("optimal threshold: ", OptimalTh)
 
first, second = interpany(predc, truth, OptimalTh)
print("first = ", first, " second = ", second)

Таким образом, весь код будет выглядеть следующим образом:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os
import random
import numpy as np
tf.keras.backend.set_floatx('float64')

# data preparing
import pandas as pd
df = pd.read_csv('путь к sign_dump.csv')

friend_users = [1,8,9,10,11,14,15,17,18,19,20,22]
alien_users = [2,3,4,5,6,7,12,16,21]

friend_df = df[df['id_user'].isin(friend_users)]
alien_df  = df[df['id_user'].isin(alien_users)]

alien_df['id_user']=0 

feed_df=friend_df.merge(alien_df,how='outer')

X = feed_df.drop(columns='id_user')
Y = feed_df['id_user']

import pylab
pylab.rcParams['figure.figsize'] = (15, 10)
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GaussianNoise

from tensorflow.keras.utils import to_categorical
dummy_y = to_categorical(Y)
from sklearn.model_selection import train_test_split

seed_value= 10
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
tf.keras.utils.set_random_seed(seed_value)

X_train,X_test,Y_train,Y_test = train_test_split(X,dummy_y)

model = Sequential()
model.add(GaussianNoise(0.15,input_shape=(144,),name="noise_layer"))
model.add(Dense(30,activation='relu',input_shape=(144,)))
model.add(Dense(dummy_y.shape[1],activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

results = model.fit(X_train,Y_train,validation_data = (X_test,Y_test),epochs=150)

model.save_weights('sign_model.h5')

def display_accuracy():
    plt.plot(results.history['accuracy'])
    plt.plot(results.history['val_accuracy'])

    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Test'], loc='upper left')
    plt.show()

def display_loss():
    plt.plot(results.history['loss'])
    plt.plot(results.history['val_loss'])

    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Test'], loc='upper left')
    plt.show()

display_accuracy()
display_loss()

import numpy as np
from scipy.optimize import minimize

def interpany(classouts,classets,classporog):
    classets = classets.applymap(bool)
    level = np.ones((classouts.shape[0],1))*classporog
    t2=classouts>level
    s1 = np.sum((t2==0)&classets)
    s2 = np.sum(t2&(~classets))
    ss1= np.sum(classets)
    p1=np.sum(s1)/np.sum(ss1)
    p2=np.sum(s2)/(t2.shape[0]*classets.shape[1]-np.sum(ss1))
    return np.array([p1,p2])

def GetMistSum(threahold, predc, truth, priority):
    first, second = interpany(predc, truth, threahold/100)
    return first + priority * second

def FindTh(predc, truth, priority):
    res = minimize(fun=GetMistSum, x0=90, args=(predc, truth, priority),
                   method='powell', options={'disp': True})
    return res.x / 100

target_df = feed_df.copy()
 
X = target_df.drop(columns='id_user')
Y = to_categorical(target_df['id_user'])
predc=pd.DataFrame(model.predict(X))
truth=pd.DataFrame(Y).applymap(round)
 
OptimalTh = FindTh(predc, truth, 10)
print("optimal threshold: ", OptimalTh)
 
first, second = interpany(predc, truth, OptimalTh)
print("first = ", first, " second = ", second)
